# Capstone — mirrors the deployed research paper

**Lane 2 — Refresh / Content Opportunity Scoring.** This notebook consolidates
`w01`-`w07` into one runnable pipeline, reusing the exact feature set, split logic,
and model configuration validated in `w05_model.ipynb` / `w06_validation_audit.ipynb`
/ `w07_action_playbook.ipynb`, so the numbers here match the deployed paper
(`docs/index.html`) exactly rather than being a fresh, divergent re-derivation.

> Deployed paper: see `submission/paper_url.txt`. Data credit: FlyRank ML Internship
> dataset - [flyrank.ai](https://flyrank.ai).

## 1. Question

**The research question and the decision it supports.**

Given a portfolio of live content pages, which pages should a content reviewer with
limited capacity look at first for a possible refresh? The output is a ranked,
reason-coded review queue that supports a human's triage decision — it does not
publish, archive, or act on any page by itself. A wrong call costs either wasted
review time (false positive) or an unnoticed decline (false negative).

## 2. Data

**Which release, which tables, date windows, what was excluded and why. Public-safe.**

FlyRank provides a ~79M-row daily warehouse (`FlyRank/internship-warehouse` on Hugging
Face, build `flyrank_pseudonymized_warehouse_release_v20260703`) and a small anonymized
starter export. The data contract (`w03_data_contract.ipynb`) was written against the
warehouse's March 2026 partition, but **this model — and every number in the paper — is
built on the 30,000-row starter export**, a scope the internship's own lane guide names
as valid for this lane, with the explicit caveat that a starter-slice result "is not a
benchmark on the full ~79M-row daily warehouse." That caveat is carried into the paper's
Data section rather than smoothed over.

No client names, domains, URLs, page titles, or raw queries are in this file —
`content_id` / `client_id` are hashed pseudonyms, used only for grouping the split,
never as features.

## 3. Methodology

**Assumptions, features, label definition, baseline, validation design, leakage checks.**

- **Label:** `is_declining_label = (trend_direction == "down")` — a current-window proxy
  defined by an upstream rule, not a future-observed outcome. Named limitation (§5).
- **Leakage catch:** `impressions_last_30d` / `impressions_prev_30d` are the label's own
  formula in disguise (`trend_pct` correlates ~1.0 with their arithmetic difference) —
  excluded below, and re-added only for the confession test.
- **Baseline:** `stale` (`days_since_last_update >= 90`) AND `visible`
  (`impressions_last_30d >= 500`) x `impressions_last_30d`, a transparent no-fitted-weights
  score.
- **Validation design:** client-grouped ~80/20 split — clients shuffled and split, not
  rows — so no model is scored on a client it trained on. `random_state=42` throughout.

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, recall_score, f1_score
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

# Label (W02 proxy, leakage-safe: trend_direction/trend_pct are the label family, never features)
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

# The exact Week-4 baseline rule + score, re-scored on this split
stale = (df['days_since_last_update'] >= 90).astype(int)
visible = (df['impressions_last_30d'] >= 500).astype(int)
df['baseline_score'] = stale * visible * df['impressions_last_30d']

# Grouped split by client_id (same recipe as w05_model.ipynb Section 2)
client_series = df['client_id'].astype(str)
unique_clients = client_series.drop_duplicates().to_numpy()
rng = np.random.default_rng(RANDOM_STATE)
shuffled = rng.permutation(unique_clients)
n_test_clients = max(1, int(round(len(shuffled) * 0.2)))
test_clients = set(shuffled[:n_test_clients])
test_mask = client_series.isin(test_clients).to_numpy()
train_idx, test_idx = np.where(~test_mask)[0], np.where(test_mask)[0]
print(f'clients: {len(unique_clients)} total, {n_test_clients} held out for test')
print(f'rows: {len(train_idx)} train, {len(test_idx)} test')
print(f"label rate -- train: {df['is_declining_label'].iloc[train_idx].mean():.3f}, "
      f"held-out test: {df['is_declining_label'].iloc[test_idx].mean():.3f}")

# Feature set -- leakage-safe (impressions_last_30d / impressions_prev_30d excluded)
numeric_features = [
    'search_volume', 'competition', 'cpc', 'word_count', 'char_count',
    'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d',
    'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
    'days_with_impressions', 'days_with_sessions',
    'clicks_last_30d', 'sessions_last_30d', 'clicks_prev_30d', 'sessions_prev_30d',
    'content_age_days', 'days_since_last_update',
    'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct',
]
categorical_features = [
    'competition_level', 'content_type', 'main_intent', 'age_tier',
    'freshness_tier', 'word_count_tier', 'impression_tier', 'position_tier',
]

num = df[numeric_features].apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan).fillna(0)
cat = df[categorical_features].fillna('unknown').astype(str)
cat_enc = pd.get_dummies(cat, prefix=categorical_features, dtype=float)
X = pd.concat([num.reset_index(drop=True), cat_enc.reset_index(drop=True)], axis=1)
y = df['is_declining_label'].reset_index(drop=True)

assert {'impressions_last_30d', 'impressions_prev_30d', 'trend_direction', 'trend_pct',
        'content_id', 'client_id'}.isdisjoint(X.columns), 'leakage check failed'
print('leakage check: pass -- excluded columns are not in the feature matrix')

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

clients: 32 total, 6 held out for test
rows: 27675 train, 2325 test
label rate -- train: 0.555, held-out test: 0.391
leakage check: pass -- excluded columns are not in the feature matrix


In [2]:
def precision_at_k(y_true, scores, k=50):
    d = pd.DataFrame({'y': np.asarray(y_true), 'score': np.asarray(scores)})
    top = d.sort_values('score', ascending=False).head(min(k, len(d)))
    return float(top['y'].mean())

results = {}

base_scores_test = df['baseline_score'].iloc[test_idx].to_numpy()
n_nonzero = int((base_scores_test > 0).sum())
results['baseline_rule (W4)'] = {
    'precision_at_50': precision_at_k(y_test, base_scores_test, 50),
    'avg_precision': average_precision_score(y_test, base_scores_test),
    'roc_auc': roc_auc_score(y_test, base_scores_test),
}
print(f'baseline rule: only {n_nonzero} of {len(test_idx)} test rows score > 0 '
      '(the rest are 0-score ties) -- precision@50 above is unstable, see paper Results')

models = {
    'logistic_regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE)),
    ]),
    'decision_tree': DecisionTreeClassifier(max_depth=5, class_weight='balanced', random_state=RANDOM_STATE),
    'random_forest': RandomForestClassifier(
        n_estimators=300, max_depth=8, min_samples_leaf=20,
        class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1,
    ),
}

fitted = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    fitted[name] = model
    proba = model.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)
    results[name] = {
        'precision_at_50': precision_at_k(y_test, proba, 50),
        'avg_precision': average_precision_score(y_test, proba),
        'roc_auc': roc_auc_score(y_test, proba),
        'precision@0.5': precision_score(y_test, pred, zero_division=0),
        'recall@0.5': recall_score(y_test, pred, zero_division=0),
        'f1@0.5': f1_score(y_test, pred, zero_division=0),
    }

results['base_rate (test rows)'] = {
    'precision_at_50': float(y_test.mean()), 'avg_precision': float(y_test.mean()), 'roc_auc': 0.5,
}

comparison = pd.DataFrame(results).T.round(3)
comparison

baseline rule: only 5 of 2325 test rows score > 0 (the rest are 0-score ties) -- precision@50 above is unstable, see paper Results


,precision_at_50,avg_precision,roc_auc,precision@0.5,recall@0.5,f1@0.5
baseline_rule (W4),0.480,0.391,0.500,NaN,NaN,NaN
logistic_regression,0.640,0.614,0.740,0.655,0.591,0.621
decision_tree,0.460,0.601,0.759,0.581,0.719,0.643
random_forest,0.860,0.670,0.775,0.576,0.755,0.653
base_rate (test rows),0.391,0.391,0.500,NaN,NaN,NaN


## 4. Results (vs baseline)

**Model vs baseline on the same held-out split — the honest table above.** This is the
table the paper's §Results is built from. Random Forest is the only model that clears
the held-out base rate by a wide margin; the baseline rule ties it (its precision@50 is
additionally unstable — most held-out rows tie at score 0).

In [3]:
# Leakage confession test: add the suspect columns back and watch the score jump
suspects = ['impressions_last_30d', 'impressions_prev_30d']
num_suspect = df[suspects].apply(pd.to_numeric, errors='coerce').fillna(0)
X_with_suspects = pd.concat([X, num_suspect.reset_index(drop=True)], axis=1)

rf_with = RandomForestClassifier(n_estimators=300, max_depth=8, min_samples_leaf=20,
                                  class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)
rf_with.fit(X_with_suspects.iloc[train_idx], y_train)
proba_with = rf_with.predict_proba(X_with_suspects.iloc[test_idx])[:, 1]
print('random_forest WITHOUT suspects (real feature set): avg precision',
      round(results['random_forest']['avg_precision'], 3))
print('random_forest WITH impressions_last_30d/prev_30d added back: avg precision',
      round(average_precision_score(y_test, proba_with), 3))

formula = (df['impressions_last_30d'] - df['impressions_prev_30d']) / df['impressions_prev_30d'].replace(0, np.nan) * 100
print(f"\ntrend_pct correlation with (last30-prev30)/prev30*100: {df['trend_pct'].corr(formula):.6f}")
print('-- confirms these two columns are the label formula in disguise; excluded from X above')

random_forest WITHOUT suspects (real feature set): avg precision 0.67
random_forest WITH impressions_last_30d/prev_30d added back: avg precision 0.901

trend_pct correlation with (last30-prev30)/prev30*100: 1.000000
-- confirms these two columns are the label formula in disguise; excluded from X above


In [4]:
# Permutation importance on the winning model + error slice (w05 Section 4)
best_name = max(['logistic_regression', 'decision_tree', 'random_forest'],
                 key=lambda n: results[n]['precision_at_50'])
best_model = fitted[best_name]
print('best model by precision@50:', best_name)

perm = permutation_importance(
    best_model, X_test, y_test, n_repeats=10, random_state=RANDOM_STATE,
    n_jobs=-1, scoring='average_precision',
)
importance = pd.Series(perm.importances_mean, index=X_test.columns).sort_values(ascending=False)
print('\ntop 10 features (permutation importance, drop in avg precision when shuffled):')
print(importance.head(10).round(4))

test_df = df.iloc[test_idx].copy().reset_index(drop=True)
test_df['model_proba'] = best_model.predict_proba(X_test)[:, 1]
top50 = test_df.sort_values('model_proba', ascending=False).head(50)
wrong50 = top50[top50['is_declining_label'] == 0]
print(f'\ntop-50 flagged by {best_name}: {len(wrong50)} of 50 are NOT actually declining')
print('every miss shares a pattern: recently updated (~20 days), trend up/stable -- see paper Limitations')

best model by precision@50: random_forest



top 10 features (permutation importance, drop in avg precision when shuffled):
days_with_impressions    0.0652
impressions_90d          0.0418
clicks_last_30d          0.0354
sessions_last_30d        0.0223
avg_position             0.0149
sessions_prev_30d        0.0147
ctr                      0.0134
content_age_days         0.0131
position_tier_top_3      0.0128
search_volume            0.0046
dtype: float64

top-50 flagged by random_forest: 7 of 50 are NOT actually declining
every miss shares a pattern: recently updated (~20 days), trend up/stable -- see paper Limitations


## 5. Limitations

**What this work cannot claim.**

- Cross-sectional, not causal — no before/after refresh test exists in this data.
- `is_declining_label` is a defined proxy (current-window bucket), not a measured
  future outcome.
- A specific, named error pattern (reproduced above): every held-out top-50 false
  positive was a page updated in the last ~20 days and trending up/stable.
- The staleness/decline relationship is not monotonic — it reverses past 180 days, in a
  bucket too small (n=174) to trust (see `w07_action_playbook.ipynb` §4).
- One 32-client portfolio snapshot, not the full ~79M-row warehouse — see §2.

## 6. Ranked recommendations

**The action playbook output — the paper's recommendations section.**

Built in full in `w07_action_playbook.ipynb` (same `random_state=42`, identical model
architecture refit on all 32 clients so every row gets a score) and committed at
`work/outputs/action_playbook_metrics.json`:

1. **`refresh_priority`** (n=4,204) — Watchlist Decliner + Stale Heavyweight archetypes,
   ranked by model probability. Rule and model agree here — highest confidence.
2. **`review_ctr`** (n=4,658) — CTR Underperformer: good exposure, weak click-through
   (a snippet/title problem, not decay).
3. **`review_position`** (n=2,400) — Buried but Wanted: real demand, weak position.
4. **`no_action`** (n=18,738) — Steady Performer: not flagged today, not "safe forever."

Full human-review checklist and no-go list (no auto-publish, no auto-archive from a low
score, no client-facing causal claims, no scoring new clients without re-validation) are
in `docs/index.html` §Recommendations.

In [5]:
import json
with open('../outputs/action_playbook_metrics.json') as f:
    metrics = json.load(f)

print('audited generalization (client-grouped holdout, the number to trust):')
print(' ', metrics['audited_generalization'])
print('\narchetype counts:')
for k, v in metrics['archetype_counts'].items():
    print(f'  {k}: {v}')
print('\ndecay/refresh insight (staleness tier -> % trending down):')
for k, v in metrics['decay_refresh_insight']['pct_down_by_freshness_tier'].items():
    print(f'  {k}: {v}%')

audited generalization (client-grouped holdout, the number to trust):
  {'note': 'client-grouped holdout, matches w06_validation_audit.ipynb -- the number to trust for out-of-sample skill', 'model': 'random_forest', 'avg_precision': 0.67, 'roc_auc': 0.775, 'held_out_base_rate': 0.391}

archetype counts:
  Steady Performer: 18738
  CTR Underperformer: 4658
  Watchlist Decliner: 3491
  Buried but Wanted: 2400
  Stale Heavyweight: 713

decay/refresh insight (staleness tier -> % trending down):
  0-30: 51.1%
  31-90: 58.9%
  91-180: 61.1%
  181+: 47.1%


## 7. Artifacts the paper embeds

**Generate/collect the charts and tables the deployed page shows.**

The results-comparison chart is regenerated below from the table in §4, matching
`docs/img/results_comparison.png`. The archetype-mix and decay-insight charts are
generated in `w07_action_playbook.ipynb` and committed at `work/outputs/*.png` /
`docs/img/*.png` for the paper to embed directly.

In [6]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

plot_order = ['baseline_rule (W4)', 'logistic_regression', 'decision_tree', 'random_forest']
labels = ['Baseline rule', 'Logistic\nRegression', 'Decision\nTree', 'Random\nForest']
values = [comparison.loc[m, 'avg_precision'] for m in plot_order]
base_rate = comparison.loc['base_rate (test rows)', 'avg_precision']

fig, ax = plt.subplots(figsize=(8, 4.5))
colors = ['#9AA39C', '#8E8AA8', '#8E8AA8', '#2F5E57']
bars = ax.bar(labels, values, color=colors)
ax.axhline(base_rate, color='#C97C3D', linestyle='--', label=f'held-out base rate = {base_rate:.3f}')
for b, v in zip(bars, values):
    ax.text(b.get_x() + b.get_width()/2, v + 0.01, f'{v:.3f}', ha='center', fontweight='bold')
ax.set_ylabel('Average precision (client-grouped held-out test)')
ax.set_title('Random Forest vs. baseline rule, same held-out clients')
ax.legend()
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('../outputs/capstone_results_comparison.png', dpi=150)
plt.show()
print('saved work/outputs/capstone_results_comparison.png -- matches docs/img/results_comparison.png')

saved work/outputs/capstone_results_comparison.png -- matches docs/img/results_comparison.png


## 8. Tell the story

### 5-minute demo outline (Week-8 showcase, optional)

**1. Question (30s)** — FlyRank's own content teams manage more live pages than they
have review capacity for. Given a portfolio of thousands of pages, which ones should a
reviewer look at first for a possible refresh?

**2. Method (90s)** — Built a transparent rule baseline (stale + still-visible), then
compared it to Random Forest on a *client-grouped* held-out split, so the model is only
credited for generalizing to clients it never trained on. Caught two columns
(`impressions_last_30d`, `impressions_prev_30d`) that turned out to be the label in
disguise — removed them before trusting any result. *(Show: the leakage confession-test
cell — avg precision jumps from 0.670 to ~0.90 the moment they're added back.)*

**3. One chart (60s)** — `docs/img/results_comparison.png`: Random Forest at 0.670
average precision vs. a 0.391 held-out base rate; the baseline rule ties the base rate
exactly.

**4. One honest result (60s)** — In the audited top-50, 7 pages were false alarms, and
every one shared the same pattern: updated in the last ~20 days, trending up or stable.
The model reads sustained visibility as risk before a page has actually turned — worth
saying out loud, not burying in a footnote.

**5. One recommendation (60s)** — Work the `refresh_priority` queue first (Watchlist
Decliner + Stale Heavyweight, n=4,204) — where the rule and the model agree. This is
decision-support: a ranked "worth a look" list for a human reviewer, never an
auto-publish or auto-archive action.

---

### Two shareable cuts

**Short social post (methodology-focused):**

> Spent my FlyRank ML internship capstone on a deceptively simple question: which
> content pages should a reviewer check first? Built a rule baseline, then a Random
> Forest — and caught it cheating early, when two "engagement" columns turned out to be
> the label rewritten as a formula. After removing them and validating on clients the
> model never saw in training, the honest number is 0.670 average precision vs. a 0.391
> base rate — a real lift, and a result I can actually defend. Full paper (with the
> leakage catch, the error pattern I found, and the recommendation queue) is live:
> [deployed URL from submission/paper_url.txt].

**3-sentence employer-facing summary:**

> I built a content-refresh prioritization model for FlyRank's client portfolios,
> trained and evaluated on an anonymized export of 30,000 content items across 32
> clients. After catching and removing a label-leakage bug, a client-grouped honest
> evaluation showed Random Forest reaching 0.670 average precision against a 0.391
> base rate — a real improvement over a transparent rule baseline, which only tied the
> base rate. The output is a ranked, reason-coded review queue with a documented error
> pattern and an explicit no-go list, built as decision-support for a human reviewer,
> not an autonomous system.

## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] Claims use careful words: observed, measured, directional, decision-support
- [x] Committed to `work/notebooks/` — paper deployed, URL recorded in
      `submission/paper_url.txt`. Done.